In [1]:
import external_checks
import fetch_data
import internal_checks
import pandas as pd

internal_data = internal_checks.get_internal_data()
external_data = fetch_data.fetch_all()
external_comparisons = external_checks.compare_against_external_data(internal_data, external_data)

print("External data")
display(pd.Series(external_data).apply(len))
print("External comparisons")
display(pd.Series(external_comparisons).apply(len))

Fetching
OpenAlex authors: 50/50
OpenAlex publications by author: 50/50
ORCID records: 28/28
OpenAlex publications by DOI: 4624/4624
Crossref publications: 4624/4624
DataCite publications: 67/67
DOI resolutions: 9/9
Semantic Scholar publications by DOI: 4624/4624
External data


dois                                    4624
openalex_authors                          50
openalex_publications_by_author           50
orcid_records                             28
openalex_publications_by_doi            4624
crossref_publications                   4557
datacite_publications                     58
doi_resolutions                            9
semantic_scholar_publications_by_doi    4393
dtype: int64

External comparisons


reference_agreement                   2
publication_coverage                 50
orcid_names                          28
orcid_recoverability                 50
semantic_scholar_author_evidence    267
affiliations                          2
year_spans                           50
dtype: int64

### Checking stored OpenAlex papers against fetched OpenAlex and Crossref (with Datacite as fallback)

In [2]:
reference_agreement = external_comparisons["reference_agreement"]


def as_percentages(table):
    return (100 * table / table.sum()).round(1)


def agreement_table(metric):
    table = pd.DataFrame({side: summary[metric] for side, summary in reference_agreement.items()})
    return as_percentages(table.fillna(0))


def year_gap_table():
    counts = {}
    for side, summary in reference_agreement.items():
        counts[side] = {
            "unavailable" if gap is None else gap: count
            for gap, count in summary["year_gaps"].items()
        }
    table = pd.DataFrame(counts).fillna(0)
    order = sorted(gap for gap in table.index if gap != "unavailable")
    if "unavailable" in table.index:
        order.append("unavailable")
    return as_percentages(table.reindex(order))


print("Title %")
display(agreement_table("title_matches"))
print("Journal %")
display(agreement_table("journal_matches"))
print("Year gap %")
display(year_gap_table())

internal_by_source = external_checks.get_publications_by_source(internal_data)
internal_counts = {source: len(dois) for source, dois in internal_by_source.items()}
internal_counts["all"] = len(set().union(*internal_by_source.values()))
fetched_counts = {
    "openalex": len(external_data["openalex_publications_by_doi"]),
    "crossref": len(external_data["crossref_publications"]),
    "datacite": len(external_data["datacite_publications"]),
}
fetched_counts["all"] = len(
    external_data["openalex_publications_by_doi"].keys()
    | external_data["crossref_publications"].keys()
    | external_data["datacite_publications"].keys()
)
doi_counts = pd.DataFrame({"internal": internal_counts, "fetched": fetched_counts}).fillna(0).astype(int)
doi_counts = doi_counts.reindex(["openalex", "crossref", "datacite", "all"])
print("Unique DOI counts for each source")
display(doi_counts)

Title %


,same source,across sources
exact,99.4,97.3
mismatch,0.3,0.2
unavailable,0.0,0.2
normalized_match,0.3,2.3


Journal %


,same source,across sources
exact,91.6,77.1
normalized_match,0.2,7.3
unavailable,7.9,9.3
mismatch,0.3,6.2


Year gap %


,same source,across sources
0,99.9,92.3
1,0.0,5.9
2,0.0,0.1
3,0.0,0.0
5,0.0,0.0
10,0.0,0.0
unavailable,0.0,1.6


Unique DOI counts for each source


,internal,fetched
openalex,4623,4624
crossref,479,4557
datacite,0,58
all,4624,4624


### Availability of stored OpenAlex DOIs in other registries

In [3]:
availability = external_checks.summarize_doi_availability(external_data)
display(pd.Series(availability, name="dois"))
registered = availability["crossref"] + availability["datacite_not_crossref"]
print(
    f"{100 * registered / sum(availability.values()):.1f}% of internal OpenAlex DOIs resolve in Crossref or Datacite "
)
print(f"{availability['datacite_not_crossref']} were found in Datacite but were missing from Crossref")

crossref                 4557
datacite_not_crossref      58
unresolved                  6
resolves_only               3
Name: dois, dtype: int64

99.8% of internal OpenAlex DOIs resolve in Crossref or Datacite 
58 were found in Datacite but were missing from Crossref


### Duplicate DOIs stored in an author's publications

In [4]:
summary = internal_checks.summarize_duplicate_dois(internal_data)
print(f"DOIs stored more than once for one author: {summary['duplicate_dois']}")
pd.Series(summary["records_per_source"]).rename_axis("duplicate_source").to_frame("Records")

DOIs stored more than once for one author: 481


,Records
duplicate_source,
openalex,481
crossref,481


### Mismatch between internal author names and fetched ORCID profiles

In [5]:
orcid_names = pd.DataFrame(external_comparisons["orcid_names"])
mismatches = orcid_names.loc[orcid_names["status"] == "mismatch"].sort_values(
    "character_similarity"
)
display(mismatches[["profile_name", "orcid_name"]].reset_index(drop=True))

print("ORCID status")
display(pd.DataFrame(external_comparisons["orcid_recoverability"])["status"].value_counts())
print(
    "Recoverable indicates the fetched OpenAlex profile has an ORCID but the internal profile does not"
)

,profile_name,orcid_name
0,Kenneth S. Suslick,Yagang Zhang
1,Philip S. Yu,Xuan Lin
2,Stephen P. Long,Rachel G Shekar
3,Andrew A. Gewirth,Jongwon Kim
4,Marshall Scott Poole,Marja Turunen
5,I. Petrov,Mikhail I. Petrov
6,Karen M. Tabb,Karen Tabb Dina
7,Boxuan Zhao,Boxuan Simen Zhao
8,Tarek Abdelzaher,Tarek F. Abdelzaher


ORCID status


status
present        28
recoverable    19
missing         3
Name: count, dtype: int64

Recoverable indicates the fetched OpenAlex profile has an ORCID but the internal profile does not


### Semantic Scholar can be used to flag profiles for manual review (merged profiles)

Each profile can be mapped to a number of distinct S2 IDs, found by matching the profile's name against the authors on its publications in S2. Some of these profiles with high counts, such as Eric R. Larson and Heidi Phillips, look like they belong to more than one person after manual review.

In [6]:
s2_evidence = pd.DataFrame(external_comparisons["semantic_scholar_author_evidence"])
s2_id_counts = s2_evidence.groupby("author_id")["semantic_scholar_author_id"].nunique()
s2_summary = pd.DataFrame(
    [
        {
            "author_id": author_id,
            "profile_name": author["profile"]["name"],
            "distinct_s2_author_ids": s2_id_counts.get(author_id, 0),
        }
        for author_id, author in internal_data.items()
    ]
).sort_values("distinct_s2_author_ids", ascending=False)
display(s2_summary[["profile_name", "distinct_s2_author_ids"]].head(15).reset_index(drop=True))

,profile_name,distinct_s2_author_ids
0,Boxuan Zhao,15
1,Eric R. Larson,15
2,Kai Luo,15
3,Axel Hoffmann,11
4,Andrew N. Miller,10
5,Elvira González de Mejı́a,9
6,Heidi Phillips,9
7,Nishant Garg,9
8,Robert K. Yu,9
9,William P. King,8


### Evidence for profiles with the highest number of S2 IDs

Limited to the first 10 per author

In [7]:
top_s2_author_ids = s2_summary.head(2)["author_id"]
s2_details = s2_evidence.loc[s2_evidence["author_id"].isin(top_s2_author_ids)].copy()
s2_details["profile_name"] = s2_details["author_id"].map(
    lambda author_id: internal_data[author_id]["profile"]["name"]
)
s2_details = (
    s2_details.sort_values(
        ["profile_name", "matched_publication_share_percent"], ascending=[True, False]
    )
    .groupby("profile_name")
    .head(10)
    .reset_index(drop=True)
)
with pd.option_context("display.max_colwidth", 120):
    display(
        s2_details[
            [
                "profile_name",
                "semantic_scholar_author_id",
                "matched_publication_share_percent",
                "sample_title",
            ]
        ]
    )

,profile_name,semantic_scholar_author_id,matched_publication_share_percent,sample_title
0,Boxuan Zhao,16269112,47.7,N6-methyladenosine Modulates Messenger RNA Translation Efficiency
1,Boxuan Zhao,49217626,20.9,Toward Safe Human Robot Collaboration by Using Multiple Kinects Based Real-Time Human Tracking
2,Boxuan Zhao,2256773390,5.8,Connectome-seq: high-throughput mapping of neuronal connectivity at single-synapse resolution via barcode sequencing.
3,Boxuan Zhao,2350328070,5.8,4D printing composite with electrically controlled local deformation
4,Boxuan Zhao,2211176765,3.5,Engineered allostery in light-regulated LOV-Turbo enables precise spatiotemporal control of proximity labeling in li...
5,Boxuan Zhao,2347749212,3.5,Formation and Chemical Structure of Carbon-13 Tracer Lignin-Carbohydrate Complexes (LCCs) During Kraft Pulping
6,Boxuan Zhao,2143737719,2.3,Seeing on the surface of Eyeballs: Non-invasive visual substitution with the electro-tactile device on cornea
7,Boxuan Zhao,2238727674,2.3,Observation of metal-organic interphase in Cu-based electrochemical CO2-to-ethanol conversion
8,Boxuan Zhao,2143736177,1.2,A network pharmacology approach to explore the mechanism of action of Yiqi Fumai lyophilized injection in the treatm...
9,Boxuan Zhao,2143737250,1.2,A facile method for constructing a superhydrophobic zinc coating on a steel surface with anti-corrosion and drag-red...


### Internal publications vs fetched OpenAlex output for each author

In [8]:
coverage = pd.DataFrame(external_comparisons["publication_coverage"])
profile_counts = pd.DataFrame(internal_checks.get_publication_claims(internal_data))
publication_counts = coverage.merge(profile_counts[["author_id", "claim"]], on="author_id")
publication_counts["profile_name"] = publication_counts["author_id"].map(
    lambda author_id: internal_data[author_id]["profile"]["name"]
)

display(
    publication_counts[
        [
            "profile_name",
            "claim",
            "external_publication_count",
            "internal_doi_count",
            "external_doi_count",
            "shared_doi_count",
        ]
    ]
    .rename(
        columns={
            "claim": "internal_profile_publications",
            "internal_doi_count": "internal_openalex_dois",
            "external_doi_count": "external_openalex_dois",
            "external_publication_count": "external_openalex_publications",
            "shared_doi_count": "internal_external_doi_overlap",
        }
    )
    .sort_values("external_openalex_publications", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(
    coverage[["internal_dois_found_in_openalex", "openalex_dois_found_internally"]]
    .describe()
    .round(2)
)

,profile_name,internal_profile_publications,external_openalex_publications,internal_openalex_dois,external_openalex_dois,internal_external_doi_overlap
0,Philip S. Yu,2668,2670,97,2500,97
1,M. Lokajı́ček,2279,2229,98,1976,98
2,Michael A. Peters,1689,1688,75,1515,75
3,Kai Luo,807,799,98,709,98
4,Laxmikant V. Kalé,906,693,89,604,89
5,Charles H. Hillman,688,685,99,676,99
6,Tarek Abdelzaher,682,682,96,598,96
7,Pedro Curi Hallal,647,647,95,550,95
8,Qingyan Chen,601,601,98,537,98
9,John W. Erdman,596,595,97,520,97


,internal_dois_found_in_openalex,openalex_dois_found_internally
count,50.0,50.00
mean,1.0,0.35
std,0.0,0.23
min,1.0,0.04
25%,1.0,0.20
50%,1.0,0.25
75%,1.0,0.45
max,1.0,0.99


### Profile author may be missing from their own publications

One reason for this is that OpenAlex author lists are capped at 100 authors.

In [9]:
# only compare publications with both OpenAlex and Crossref entries
author_lists = pd.DataFrame(
    external_checks.compare_author_list_lengths(internal_data, external_data)
)
print("Number of authors in for a DOI in publications.json")
display(
    author_lists[["internal_openalex", "internal_crossref", "external_crossref"]]
    .describe()
    .round(1)
)

pd.crosstab(
    author_lists["profile_author_in_internal_openalex"],
    author_lists["profile_author_in_internal_crossref"],
)

Number of authors in for a DOI in publications.json


,internal_openalex,internal_crossref,external_crossref
count,481.0,481.0,481.0
mean,15.5,94.5,94.5
std,28.8,426.1,426.1
min,1.0,0.0,0.0
25%,3.0,3.0,3.0
50%,5.0,5.0,5.0
75%,9.0,9.0,9.0
max,100.0,5112.0,5112.0


profile_author_in_internal_crossref,False,True
profile_author_in_internal_openalex,,
False,0,28
True,6,447


### Verification status

In [10]:
print("openAccessPdf = True almost always means deep_verified = True")
deep_buckets = [
    "verified_with_pdf",
    "verified_without_pdf",
    "unverified_with_pdf",
    "unverified_empty",
    "deep_verification_absent",
]
display(
    pd.Series(internal_checks.summarize_deep_verifications(internal_data))
    .reindex(deep_buckets, fill_value=0)
    .rename("deep_verification")
)

verification_details = pd.DataFrame(internal_checks.compare_profile_verifications(internal_data))

print("Profile link status")
display(
    pd.crosstab(verification_details["verification_status"], verification_details["source"])
    .reindex(["verified", "unverified", "missing"])
    .dropna(how="all")
    .astype(int)
)

print("URLs matching profile regex in the author's broad_impact file")
display(
    pd.crosstab(verification_details["matching_profile_url"], verification_details["source"])
    .rename(index={False: "absent", True: "present"})
    .reindex(["present", "absent"])
)

print("Verification reasons")
display(
    verification_details.dropna(subset=["reason"])
    .groupby(["source", "reason"])
    .size()
    .rename("profiles")
    .reset_index()
)

openAccessPdf = True almost always means deep_verified = True


verified_with_pdf           2343
verified_without_pdf          19
unverified_with_pdf            0
unverified_empty            3009
deep_verification_absent       0
Name: deep_verification, dtype: int64

Profile link status


source,google_scholar,linkedin,researchgate
verification_status,,,
unverified,50,0,0
missing,0,50,50


URLs matching profile regex in the author's broad_impact file


source,google_scholar,linkedin,researchgate
matching_profile_url,,,
present,26,6,16
absent,24,44,34


Verification reasons


,source,reason,profiles
0,google_scholar,HTTP 429,50


### Titles with uncleaned markup

In [11]:
markup = internal_checks.get_titles_with_markup(internal_data)
display(
    pd.Series({markup_type: len(titles) for markup_type, titles in markup.items()}, name="titles")
)

examples = markup["html"][:2] + markup["mml"][:2]
with pd.option_context("display.max_colwidth", 120):
    display(pd.DataFrame(examples)[["author_id", "title"]])

mml      65
html    351
Name: titles, dtype: int64

,author_id,title
0,A5003524720,Evolution of transcript modification by <i>N</i><sup>6</sup>-methyladenosine in primates
1,A5003524720,Our views of dynamic <i>N</i><sup>6</sup>-methyladenosine RNA methylation
2,A5046529785,"Quantization of fractional corner charge in <mml:math xmlns:mml=""http://www.w3.org/1998/Math/MathML""><mml:msub><mml:..."
3,A5046529785,"Orbitronics: The Intrinsic Orbital Current in<mml:math xmlns:mml=""http://www.w3.org/1998/Math/MathML"" display=""inlin..."


### Doubled journal names

In [12]:
pd.DataFrame(internal_checks.get_doubled_journals(internal_data)).value_counts().reset_index(
    name="paper_count"
)

,author_id,journal,paper_count
0,A5046529785,Physical review. B./Physical review. B,23
1,A5112283538,Physical review. D/Physical review. D.,20
2,A5019834075,Physical review. B./Physical review. B,6
3,A5036508375,Physical review. D/Physical review. D.,4
4,A5013197591,Physical review. D/Physical review. D.,2
5,A5001475799,Physical review. B./Physical review. B,1
6,A5004324846,Physical review. D/Physical review. D.,1
7,A5072717764,Physical review. B./Physical review. B,1
8,A5074295593,Advances in ecological research/Advances in Ec...,1


### Impossible year spans in internal OpenAlex data are also present in fetched OpenAlex data

In [13]:
spans = pd.json_normalize(external_comparisons["year_spans"]).sort_values(
    "internal.year_span", ascending=False
)
spans.set_index("author_id")[["internal.year_span", "external.year_span"]].head()

,internal.year_span,external.year_span
author_id,,
A5065601087,341,341
A5082046729,109,109
A5023340512,87,87
A5027180367,76,76
A5074295593,74,74


### Affiliation just tracks the first value in last_known_institutions

In [14]:
affiliations = external_comparisons["affiliations"]

print("Match position of the internal affiliation within last_known_institutions from OpenAlex")
display(pd.Series(affiliations["match_positions"], name="authors").sort_index())

print("Last known institutions for authors fetched from OpenAlex")
display(
    pd.Series(affiliations["affiliation_counts"]["last_known_institutions"])
    .rename_axis("last_known_institutions")
    .reset_index(name="authors")
    .sort_values("last_known_institutions", ascending=False)
    .reset_index(drop=True)
)

print("Historical affiliations for authors fetched from OpenAlex")
display(
    pd.Series(affiliations["affiliation_counts"]["historical_affiliations"])
    .rename_axis("historical_affiliations")
    .reset_index(name="authors")
    .sort_values("historical_affiliations", ascending=False)
    .reset_index(drop=True)
    .head(10)
)

Match position of the internal affiliation within last_known_institutions from OpenAlex


1    50
Name: authors, dtype: int64

Last known institutions for authors fetched from OpenAlex


,last_known_institutions,authors
0,66,1
1,26,1
2,8,1
3,6,3
4,5,4
5,4,3
6,3,9
7,2,11
8,1,17


Historical affiliations for authors fetched from OpenAlex


,historical_affiliations,authors
0,151,1
1,133,1
2,91,1
3,85,1
4,69,1
5,68,1
6,61,1
7,56,3
8,52,1
9,51,1
